---
# GEPA Assay Extraction - Stage 0 Pilot

**Project:** AI6129 Pathogen Tracking System  
**Version:** 1.5  
**Date:** January 2026

## Purpose
This notebook implements GEPA-based prompt optimisation for assay extraction from PubMed XML articles.
Stage 0 is a pilot to validate the approach using 10%/20%/30% training splits.

## Research Questions Addressed
- RQ2: How much ground truth data is needed for GEPA training?
- RQ3: Which models perform best for pathogen information extraction?

## Model Configuration
- **Extraction Model:** Claude Haiku 4.5 (cost-effective for inference)
- **Reflection Model:** Claude Sonnet 4 (higher capability for GEPA reflection)

## Changes in v1.5
- Fixed compare_extractions() to use dictionaries instead of sets for storing items
- Root cause: sets require hashable elements, but assay values can be dicts/lists
- Solution: use dict with item_key as key, storing (isolate_id, assay_type, value) as value

## Changes in v1.4
- Added provide_traceback=True to Evaluate calls for detailed error tracebacks
- Added explicit hashability validation for all Example fields before creation
- Added type coercion to ensure all Example fields are primitive strings
- Wrapped Example creation in try-except to catch and log hashability errors

## Changes in v1.3
- Added comprehensive logging configuration: detailed logs to file, errors only to console
- Added GEPA log_dir parameter for elaborate GEPA-specific logging and checkpoint resuming
- Fixed persistent 'unhashable type: list' error by ensuring cache=False is properly applied

## Changes in v1.2
- Fixed unhashable type error: ground_truth now serialised to JSON string for DSPy Example hashability
- Added deserialisation in metric function to restore ground_truth dictionary
- Disabled DSPy caching (cache=False) to prevent hashability errors with complex Prediction objects

## Changes in v1.1
- Fixed XML mapping to handle {PMCID}_{Date}.xml filename format
- Aligned token estimation and truncation with accession extraction code
---

---
## Section 1: Setup and Installation
---

In [1]:
!pip install dspy --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 312.3/312.3 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 7.0 MB/s eta 0:00:00


In [2]:
import json
import logging
import os
import re
import time
from dataclasses import dataclass, field
from datetime import datetime
from enum import Enum
from pathlib import Path
from typing import Dict, List, Set, Tuple, Optional, Any, Union

from google.colab import drive
from google.colab import userdata
import xml.etree.ElementTree as ET

import dspy

---
## Section 2: Configuration Constants
---

In [3]:
# =============================================================================
# DSPY MODEL STRINGS
# =============================================================================
DSPY_MODEL_STRINGS = {
    "claude-haiku-4.5": "anthropic/claude-haiku-4-5-20251001",
    "claude-sonnet-4": "anthropic/claude-sonnet-4-20250514",
    "claude-sonnet-4-bedrock": "bedrock/apac.anthropic.claude-sonnet-4-20250514-v1:0",
    "amazon-nova-pro": "bedrock/apac.amazon.nova-pro-v1:0",
}

# =============================================================================
# MODEL PRICING (USD per 1M tokens)
# =============================================================================
MODEL_PRICING = {
    "claude-haiku-4.5": {"input": 1.00, "output": 5.00},
    "claude-sonnet-4": {"input": 3.00, "output": 15.00},
    "amazon-nova-pro": {"input": 0.80, "output": 3.20},
}

# =============================================================================
# ASSAY TYPES
# =============================================================================
BACTERIAL_ASSAY_TYPES = [
    "serotype", "mlst", "cgmlst", "spi", "ast_data",
    "amr", "plasmid", "pfge", "toxin", "phage_type",
    "snp", "virulence_genes"
]

VIRAL_ASSAY_TYPES = [
    "genotype", "subgenotype", "viral_load",
    "sequencing_region", "phylogenetic_clade", "serology"
]

UNIVERSAL_ASSAY_TYPES = ["snp", "virulence_factors"]

ALL_ASSAY_TYPES = list(set(BACTERIAL_ASSAY_TYPES + VIRAL_ASSAY_TYPES + UNIVERSAL_ASSAY_TYPES))

# =============================================================================
# GEPA CONFIGURATION
# =============================================================================
GEPA_CONFIG = {
    "num_threads": 2,
    "auto": "light",
    "track_stats": True,
    "use_merge": False,
}

# =============================================================================
# TEXT PROCESSING CONFIGURATION
# =============================================================================
DEFAULT_MAX_TOKENS = 100000

---
## Section 3: Type Definitions
---

In [4]:
class ExtractionCategory(Enum):
    """Classification categories for extracted vs ground truth comparison."""
    TRUE_POSITIVE = "TP"
    FALSE_POSITIVE = "FP"
    UNCERTAIN_POSITIVE = "UP"
    FALSE_NEGATIVE = "FN"


@dataclass
class GEPAExperimentConfig:
    """Configuration for a GEPA optimisation experiment."""
    experiment_id: str
    split_name: str
    extraction_model_id: str
    reflection_model_id: str
    num_threads: int
    auto_budget: str
    output_directory: str
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())


@dataclass
class EvaluationMetrics:
    """Evaluation metrics for assay extraction."""
    pmcid: str
    true_positives: int
    false_positives: int
    uncertain_positives: int
    false_negatives: int
    conservative_f1: float
    primary_f1: float
    optimistic_f1: float
    details: Dict[str, List[str]]

---
## Section 4: Path Configuration and Drive Mount
---

In [5]:
# Mount Google Drive
drive.mount('/content/drive', force_remount=False)

# =============================================================================
# PATH CONFIGURATION
# =============================================================================
BASE_DIR = Path("/content/drive/MyDrive/AI6129")
GROUND_TRUTH_DIR = BASE_DIR / "ground_truth"
XML_DIR = BASE_DIR / "xml"
STRATIFIED_SPLITS_FILE = BASE_DIR / "assay" / "validation_splits" / "assay_tadp_gepa_optimised_splits.json"

OUTPUT_DIR = BASE_DIR / "assay" / "gepa_experiments"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Base directory: {BASE_DIR}")
print(f"Ground truth directory exists: {GROUND_TRUTH_DIR.exists()}")
print(f"XML directory exists: {XML_DIR.exists()}")
print(f"Stratified splits file exists: {STRATIFIED_SPLITS_FILE.exists()}")
print(f"Output directory: {OUTPUT_DIR}")

Mounted at /content/drive
Base directory: /content/drive/MyDrive/AI6129
Ground truth directory exists: True
XML directory exists: True
Stratified splits file exists: True
Output directory: /content/drive/MyDrive/AI6129/assay/gepa_experiments


In [6]:
# Load API key from Colab secrets
try:
    os.environ["ANTHROPIC_API_KEY"] = userdata.get('CLAUDE_API_KEY')
    print("Anthropic API key loaded from Colab secrets")
except Exception as e:
    print(f"Warning: Could not load API key from secrets: {e}")
    print("Please set ANTHROPIC_API_KEY manually")

Anthropic API key loaded from Colab secrets


---
## Section 4b: Logging Configuration

Configure logging to write detailed debug info to file whilst showing only errors on console.
This helps debug 'unhashable type' errors without flooding the screen with article text.
---

In [7]:
# =============================================================================
# LOGGING CONFIGURATION
# Configure logging: detailed to file, errors only to console
# =============================================================================

# Create log directory
LOG_DIR = OUTPUT_DIR / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

# GEPA-specific log directory (for GEPA's internal logging and checkpoints)
GEPA_LOG_DIR = OUTPUT_DIR / "gepa_internal_logs"
GEPA_LOG_DIR.mkdir(parents=True, exist_ok=True)

# Generate log filename with timestamp
log_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
LOG_FILE = LOG_DIR / f"gepa_stage0_debug_{log_timestamp}.log"

# Create file handler for detailed logging
file_handler = logging.FileHandler(LOG_FILE, mode='w', encoding='utf-8')
file_handler.setLevel(logging.DEBUG)
file_handler.setFormatter(logging.Formatter(
    '%(asctime)s - %(name)s - %(levelname)s - %(message)s'
))

# Create console handler for errors only
console_handler = logging.StreamHandler()
console_handler.setLevel(logging.ERROR)  # Only errors to console
console_handler.setFormatter(logging.Formatter(
    '%(levelname)s: %(message)s'
))

# Configure DSPy logger
dspy_logger = logging.getLogger("dspy")
dspy_logger.setLevel(logging.DEBUG)
dspy_logger.handlers.clear()  # Remove any existing handlers
dspy_logger.addHandler(file_handler)
dspy_logger.addHandler(console_handler)

# Also configure the root logger for any other libraries
root_logger = logging.getLogger()
root_logger.setLevel(logging.WARNING)

print(f"Logging configured:")
print(f"  - Debug log file: {LOG_FILE}")
print(f"  - GEPA internal logs: {GEPA_LOG_DIR}")
print(f"  - Console shows: ERROR level only")

Logging configured:
  - Debug log file: /content/drive/MyDrive/AI6129/assay/gepa_experiments/logs/gepa_stage0_debug_20260126_064706.log
  - GEPA internal logs: /content/drive/MyDrive/AI6129/assay/gepa_experiments/gepa_internal_logs
  - Console shows: ERROR level only


---
## Section 5: Data Loading Functions
---

In [8]:
def load_stratified_splits(splits_filepath: str) -> Dict[str, Any]:
    """Load the stratified splits JSON file."""
    with open(splits_filepath, 'r', encoding='utf-8') as f:
        splits_data = json.load(f)

    print("Loaded stratified splits:")
    print(f"  Total documents: {splits_data['metadata']['total_documents']}")
    print(f"  Holdout size: {splits_data['metadata']['holdout_size']}")
    print(f"  Validation size: {splits_data['metadata']['validation_size']}")
    print(f"  Training pool size: {splits_data['metadata']['training_pool_size']}")
    print(f"  Split 10% size: {splits_data['metadata']['split_10_size']}")
    print(f"  Split 20% size: {splits_data['metadata']['split_20_size']}")
    print(f"  Split 30% size: {splits_data['metadata']['split_30_size']}")

    return splits_data


def get_split_pmcids(splits_data: Dict[str, Any], split_name: str) -> List[str]:
    """Get list of PMCIDs for a specific split."""
    if split_name in ["split_10", "split_20", "split_30"]:
        return splits_data["split_indices"][split_name]
    elif split_name == "validation_set":
        return splits_data["validation_set"]
    elif split_name == "holdout_set":
        return splits_data["holdout_set"]
    else:
        raise ValueError(f"Unknown split name: {split_name}")


def load_ground_truth(pmcid: str, ground_truth_dir: Path) -> Optional[Dict[str, Any]]:
    """Load ground truth JSON for a specific PMCID."""
    gt_filepath = ground_truth_dir / f"{pmcid}.json"

    if not gt_filepath.exists():
        print(f"Warning: Ground truth file not found: {gt_filepath}")
        return None

    with open(gt_filepath, 'r', encoding='utf-8') as f:
        return json.load(f)


def load_pmcid_to_xml_mapping(xml_directory: str, pmcid_list: List[str]) -> Dict[str, str]:
    """
    Create mapping of PMCID to XML file paths.

    Handles XML files with naming convention: {PMCID}_{Date}.xml
    e.g., PMC4672624_20251201.xml
    """
    xml_dir = Path(xml_directory)
    mapping = {}
    pmcid_set = set(pmcid_list)

    for xml_file in xml_dir.glob("*.xml"):
        # Extract PMCID from filename (handles PMCID_Date.xml format)
        filename = xml_file.stem
        pmcid = filename.split('_')[0]

        if pmcid in pmcid_set:
            mapping[pmcid] = str(xml_file)

    # Report missing files
    found_pmcids = set(mapping.keys())
    missing = pmcid_set - found_pmcids

    if missing:
        print(f"Warning: {len(missing)} XML files not found: {list(missing)[:5]}...")

    print(f"Built XML mapping for {len(mapping)}/{len(pmcid_list)} PMCIDs")
    return mapping

---
## Section 6: XML Processing Functions
---

In [9]:
def load_xml_content(xml_filepath: str) -> str:
    """Load raw XML content from file."""
    with open(xml_filepath, 'r', encoding='utf-8') as f:
        return f.read()


def extract_text_content(element: ET.Element) -> str:
    """Extract all text content from an XML element."""
    texts = []

    if element.text:
        texts.append(element.text.strip())

    for child in element:
        texts.append(extract_text_content(child))
        if child.tail:
            texts.append(child.tail.strip())

    return " ".join(t for t in texts if t)


def extract_article_sections(raw_xml: str) -> Dict[str, str]:
    """Parse XML and extract content by section."""
    sections = {}

    try:
        root = ET.fromstring(raw_xml)
    except ET.ParseError:
        return {"full_text": raw_xml[:50000]}

    # Extract title
    title_elem = root.find(".//article-title")
    if title_elem is not None:
        sections["title"] = extract_text_content(title_elem)

    # Extract abstract
    abstract_elem = root.find(".//abstract")
    if abstract_elem is not None:
        sections["abstract"] = extract_text_content(abstract_elem)

    # Extract body sections
    body_elem = root.find(".//body")
    if body_elem is not None:
        for sec in body_elem.findall(".//sec"):
            sec_title_elem = sec.find("title")
            sec_title = extract_text_content(sec_title_elem) if sec_title_elem is not None else "unnamed_section"
            sec_content = extract_text_content(sec)
            sections[sec_title] = sec_content

    return sections


def extract_tables_from_xml(raw_xml: str) -> List[str]:
    """Extract tables from XML and convert to text format."""
    tables = []

    try:
        root = ET.fromstring(raw_xml)
    except ET.ParseError:
        return tables

    for table_wrap in root.findall(".//table-wrap"):
        label_elem = table_wrap.find("label")
        caption_elem = table_wrap.find("caption")

        label = extract_text_content(label_elem) if label_elem is not None else "Table"
        caption = extract_text_content(caption_elem) if caption_elem is not None else ""

        table_content = convert_table_to_text(table_wrap)
        formatted_table = f"{label}: {caption}\n{table_content}"
        tables.append(formatted_table)

    return tables


def convert_table_to_text(table_element: ET.Element) -> str:
    """Convert XML table element to readable text format."""
    rows = []

    # Process header
    thead = table_element.find(".//thead")
    if thead is not None:
        for header_row in thead.findall(".//tr"):
            cells = [extract_text_content(cell) for cell in header_row.findall(".//th")]
            if cells:
                rows.append("|".join(cells))
        if rows:
            rows.append("-" * 40)

    # Process body rows
    tbody = table_element.find(".//tbody")
    if tbody is not None:
        for body_row in tbody.findall(".//tr"):
            cells = [extract_text_content(cell) for cell in body_row.findall(".//td")]
            if cells:
                rows.append("|".join(cells))

    return "\n".join(rows)


def estimate_token_count(text: str) -> int:
    """Estimate token count (approximately 4 characters per token)."""
    return len(text) // 4


def truncate_text_smart(text: str, max_tokens: int) -> str:
    """
    Truncate text preserving beginning and end.

    Keeps 70% from the front and 30% from the back to preserve
    both introduction/methods and results/conclusions.
    """
    max_chars = max_tokens * 4

    if len(text) <= max_chars:
        return text

    front_chars = int(max_chars * 0.7)
    back_chars = max_chars - front_chars - 50

    front_text = text[:front_chars]
    back_text = text[-back_chars:]

    return f"{front_text}\n\n[.... CONTENT TRUNCATED .....]\n\n{back_text}"


def compose_llm_input(sections: Dict[str, str], tables: List[str]) -> str:
    """Compose sections and tables into LLM input text."""
    parts = []

    for section_name, section_content in sections.items():
        parts.append(f"[SECTION: {section_name.upper()}]")
        parts.append(section_content)
        parts.append("")

    if tables:
        parts.append("[TABLES]")
        for idx, table_content in enumerate(tables):
            parts.append(f"[TABLE {idx + 1}]")
            parts.append(table_content)
            parts.append("")

    return "\n".join(parts)


def prepare_article_for_extraction(
    pmcid: str,
    xml_filepath: str,
    max_tokens: int = DEFAULT_MAX_TOKENS
) -> Tuple[str, str, int]:
    """
    Prepare article content for assay extraction.

    Returns:
        Tuple of (composed_text, raw_xml, token_estimate)
    """
    raw_xml = load_xml_content(xml_filepath)
    sections = extract_article_sections(raw_xml)
    tables = extract_tables_from_xml(raw_xml)

    extracted_text = compose_llm_input(sections, tables)
    token_estimate = estimate_token_count(extracted_text)

    # Truncate if exceeds max tokens
    if token_estimate > max_tokens:
        extracted_text = truncate_text_smart(extracted_text, max_tokens)
        token_estimate = max_tokens

    return extracted_text, raw_xml, token_estimate

---
## Section 7: Ground Truth Processing
---

In [10]:
def flatten_ground_truth_assays(gt_data: Dict[str, Any]) -> Dict[str, Dict[str, Any]]:
    """
    Flatten ground truth into standardised format for comparison.

    Returns:
        Dictionary mapping isolate_id -> {assay_type: value}
        For documents without isolate IDs, uses "NO_ISOLATE_ID" as key
    """
    flattened = {}
    assay_fields = BACTERIAL_ASSAY_TYPES + VIRAL_ASSAY_TYPES

    # Process isolates_with_linking
    if "isolates_with_linking" in gt_data:
        for isolate in gt_data["isolates_with_linking"]:
            isolate_id = isolate.get("isolate_id", "UNKNOWN")
            assays = {}
            for field in assay_fields:
                if field in isolate and isolate[field]:
                    assays[field] = isolate[field]
            if assays:
                flattened[isolate_id] = assays

    # Process isolate_without_linking
    if "isolate_without_linking" in gt_data:
        for idx, isolate in enumerate(gt_data["isolate_without_linking"]):
            isolate_id = f"UNLINKED_{idx}"
            assays = {}
            for field in assay_fields:
                if field in isolate and isolate[field]:
                    assays[field] = isolate[field]
            if assays:
                flattened[isolate_id] = assays

    # Process no_isolates_only_assayinformation
    if "no_isolates_only_assayinformation" in gt_data:
        nioai = gt_data["no_isolates_only_assayinformation"]
        if nioai:
            assays = {}
            for field in assay_fields:
                if field in nioai and nioai[field]:
                    assays[field] = nioai[field]
            if assays:
                flattened["NO_ISOLATE_ID"] = assays

    return flattened


def count_total_assay_data_points(flattened_gt: Dict[str, Dict[str, Any]]) -> int:
    """Count total assay data points in flattened ground truth."""
    total = 0
    for isolate_id, assays in flattened_gt.items():
        for assay_type, value in assays.items():
            if isinstance(value, list):
                total += len(value)
            elif value:
                total += 1
    return total

---
## Section 8: DSPy Signature and Module Definition
---

In [11]:
class AssayExtractionSignature(dspy.Signature):
    """
    Extract assay information from a PubMed article.

    BACTERIAL ASSAYS (Salmonella, E. coli, Campylobacter, Listeria, etc.):
    - MLST: Multi-Locus Sequence Typing (sequence types, allele profiles)
    - cgMLST: Core genome MLST (extended typing with hundreds of loci)
    - AST: Antimicrobial Susceptibility Testing (MIC values, interpretations)
    - AMR: Antimicrobial Resistance genes (gene names, mechanisms)
    - Serotype: Serological typing (O:H antigens, serovar names)
    - Plasmid: Plasmid typing (incompatibility groups, replicon types)
    - PFGE: Pulsed-Field Gel Electrophoresis (pattern designations)
    - SPI: Salmonella Pathogenicity Islands (SPI-1 through SPI-5)
    - Toxin: Toxin genes (stx1, stx2, enterotoxins, etc.)
    - Phage_Type: Bacteriophage typing results

    VIRAL ASSAYS (Hepatitis A, Hepatitis E, Norovirus, etc.):
    - Genotype: Viral genotype classification
    - Subgenotype: Viral subgenotype
    - Viral_Load: Quantitative viral RNA/DNA
    - Sequencing_Region: Genomic region sequenced
    - Phylogenetic_Clade: Cluster or clade assignment
    - Serology: Serological markers

    UNIVERSAL ASSAYS:
    - SNP: Single Nucleotide Polymorphisms
    - Virulence_Factors: Virulence genes or proteins detected

    Return JSON mapping isolate IDs to their assay results.
    If no explicit isolate IDs exist, use "NO_ISOLATE_ID" as the key.
    Only include assays explicitly reported in the article.
    """

    article_text: str = dspy.InputField(
        desc="Full text of the biomedical research article extracted from XML format"
    )

    assay_info: str = dspy.OutputField(
        desc="JSON object mapping each isolate ID to its assay results. "
             "Format: {isolate_id: {assay_type: value, ...}, ...}. "
             "Use 'NO_ISOLATE_ID' if article has assay data without specific isolate identifiers."
    )


class AssayExtractor(dspy.Module):
    """DSPy module for assay extraction with chain-of-thought reasoning."""

    def __init__(self):
        super().__init__()
        self.predictor = dspy.ChainOfThought(AssayExtractionSignature)

    def forward(self, article_text: str) -> dspy.Prediction:
        """Extract assay information from article text."""
        prediction = self.predictor(article_text=article_text)
        return prediction

---
## Section 9: Extraction Result Parsing and Validation
---

In [12]:
def parse_extraction_output(raw_output: str) -> Dict[str, Dict[str, Any]]:
    """Parse the model's JSON output into structured format."""
    if not raw_output:
        return {}

    # Try direct JSON parse
    try:
        return json.loads(raw_output)
    except json.JSONDecodeError:
        pass

    # Try to find JSON within markdown code blocks
    json_pattern = r'```(?:json)?\s*([\s\S]*?)```'
    matches = re.findall(json_pattern, raw_output)
    for match in matches:
        try:
            return json.loads(match.strip())
        except json.JSONDecodeError:
            continue

    # Try to find JSON object pattern
    brace_pattern = r'\{[\s\S]*\}'
    match = re.search(brace_pattern, raw_output)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass

    print(f"Warning: Could not parse JSON from output: {raw_output[:200]}...")
    return {}


def normalise_assay_value(value: Any) -> str:
    """Normalise an assay value for comparison."""
    if value is None:
        return ""
    if isinstance(value, list):
        return str(sorted([str(v).lower().strip() for v in value]))
    return str(value).lower().strip()


def verify_value_in_text(value: Any, article_text: str) -> bool:
    """
    Check if an extracted value appears in the article text.
    Used to distinguish FP (hallucination) from UP (in text but not GT).
    """
    article_lower = article_text.lower()

    if value is None:
        return False

    if isinstance(value, list):
        for item in value:
            if str(item).lower() in article_lower:
                return True
        return False

    if isinstance(value, dict):
        for v in value.values():
            if verify_value_in_text(v, article_text):
                return True
        return False

    return str(value).lower() in article_lower

---
## Section 10: Evaluation Metrics with UP Classification
---

In [13]:
def compare_extractions(
    extracted: Dict[str, Dict[str, Any]],
    ground_truth: Dict[str, Dict[str, Any]],
    article_text: str
) -> Dict[str, List[str]]:
    """
    Compare extracted assays against ground truth with UP classification.

    Returns:
        Dictionary with keys: TP, FP, UP, FN, each containing list of descriptions

    Note: Uses dicts instead of sets to avoid hashability issues with complex values.
    """
    results = {
        "TP": [],  # True Positive: in both extracted and GT
        "FP": [],  # False Positive: in extracted, not in GT, not in text
        "UP": [],  # Uncertain Positive: in extracted, not in GT, but IS in text
        "FN": []   # False Negative: in GT but not extracted
    }

    # Build dict of extracted items (using item_key as key to avoid hashability issues) #changed v1.5
    extracted_items = {}  #changed - use dict instead of set
    for isolate_id, assays in extracted.items():
        for assay_type, value in assays.items():
            if value is not None:
                item_key = f"{isolate_id}|{assay_type}|{normalise_assay_value(value)}"
                extracted_items[item_key] = (isolate_id, assay_type, value)  #changed - store as dict entry

    # Build dict of GT items #changed v1.5
    gt_items = {}  #changed - use dict instead of set
    for isolate_id, assays in ground_truth.items():
        for assay_type, value in assays.items():
            if value is not None:
                item_key = f"{isolate_id}|{assay_type}|{normalise_assay_value(value)}"
                gt_items[item_key] = (isolate_id, assay_type, value)  #changed - store as dict entry

    # Compare using dict keys (which are strings, hence hashable) #changed v1.5
    extracted_keys = set(extracted_items.keys())
    gt_keys = set(gt_items.keys())

    # True Positives: in both
    for item_key in extracted_keys & gt_keys:  #changed - use set intersection
        isolate_id, assay_type, value = extracted_items[item_key]
        results["TP"].append(f"{isolate_id}:{assay_type}={value}")

    # Check extracted items not in GT
    for item_key in extracted_keys - gt_keys:  #changed - use set difference
        isolate_id, assay_type, value = extracted_items[item_key]
        desc = f"{isolate_id}:{assay_type}={value}"
        if verify_value_in_text(value, article_text):
            results["UP"].append(desc)  # Uncertain Positive
        else:
            results["FP"].append(desc)  # False Positive (hallucination)

    # False Negatives: in GT but not extracted
    for item_key in gt_keys - extracted_keys:  #changed - use set difference
        isolate_id, assay_type, value = gt_items[item_key]
        results["FN"].append(f"{isolate_id}:{assay_type}={value}")

    return results


def calculate_f1_metrics(comparison: Dict[str, List[str]]) -> Tuple[float, float, float]:
    """
    Calculate three F1 metrics based on comparison results.

    Returns:
        Tuple of (conservative_f1, primary_f1, optimistic_f1)
    """
    tp = len(comparison["TP"])
    fp = len(comparison["FP"])
    up = len(comparison["UP"])
    fn = len(comparison["FN"])

    # Conservative F1: UP counted as FP
    c_prec = tp / (tp + fp + up) if (tp + fp + up) > 0 else 0.0
    c_rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    conservative_f1 = (2 * c_prec * c_rec / (c_prec + c_rec)) if (c_prec + c_rec) > 0 else 0.0

    # Primary F1: UP excluded
    p_prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    p_rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    primary_f1 = (2 * p_prec * p_rec / (p_prec + p_rec)) if (p_prec + p_rec) > 0 else 0.0

    # Optimistic F1: UP counted as TP
    o_prec = (tp + up) / (tp + fp + up) if (tp + fp + up) > 0 else 0.0
    o_rec = (tp + up) / (tp + up + fn) if (tp + up + fn) > 0 else 0.0
    optimistic_f1 = (2 * o_prec * o_rec / (o_prec + o_rec)) if (o_prec + o_rec) > 0 else 0.0

    return conservative_f1, primary_f1, optimistic_f1

---
## Section 11: GEPA Metric with Feedback
---

In [14]:
def generate_detailed_feedback(
    comparison: Dict[str, List[str]],
    ground_truth: Dict[str, Dict[str, Any]]
) -> str:
    """Generate detailed feedback text for GEPA reflection."""
    lines = []

    tp_count = len(comparison["TP"])
    fp_count = len(comparison["FP"])
    up_count = len(comparison["UP"])
    fn_count = len(comparison["FN"])
    total_gt = tp_count + fn_count

    if tp_count == total_gt and fp_count == 0:
        lines.append(f"Excellent extraction! All {tp_count} ground truth assays were correctly identified with no false positives.")
    else:
        lines.append(f"Extraction summary: {tp_count}/{total_gt} correct, {fp_count} hallucinations, {up_count} uncertain, {fn_count} missed.")

        if comparison["TP"]:
            tp_sample = ', '.join(comparison['TP'][:5])
            lines.append(f"\nCorrectly extracted ({tp_count}): {tp_sample}{'...' if len(comparison['TP']) > 5 else ''}")

        if comparison["FP"]:
            fp_sample = ', '.join(comparison['FP'][:5])
            lines.append(f"\nFalse positives - NOT in article ({fp_count}): {fp_sample}{'...' if len(comparison['FP']) > 5 else ''}")
            lines.append("Think about how to avoid extracting information not present in the source text.")

        if comparison["UP"]:
            up_sample = ', '.join(comparison['UP'][:5])
            lines.append(f"\nUncertain positives - found in text but not in GT ({up_count}): {up_sample}{'...' if len(comparison['UP']) > 5 else ''}")

        if comparison["FN"]:
            fn_sample = ', '.join(comparison['FN'][:5])
            lines.append(f"\nMissed extractions ({fn_count}): {fn_sample}{'...' if len(comparison['FN']) > 5 else ''}")
            lines.append("Think about how to identify these assay types in the article text.")

    return "\n".join(lines)


def assay_metric_with_feedback(
    example: dspy.Example,
    pred: dspy.Prediction,
    trace: Any = None,
    pred_name: Optional[str] = None,
    pred_trace: Any = None
) -> Union[float, dspy.Prediction]:
    """
    GEPA-compatible metric for assay extraction.

    Parameters:
        example: DSPy Example with pmcid, article_text, ground_truth
        pred: Model prediction with assay_info field
        trace: Optional trace information
        pred_name: If provided, returns Prediction with feedback
        pred_trace: Optional prediction trace

    Returns:
        If pred_name is None: float score (primary F1)
        If pred_name provided: dspy.Prediction(score=float, feedback=str)
    """
    # Deserialise ground_truth from JSON string (was serialised for hashability) #changed
    ground_truth = json.loads(example.ground_truth)  #changed - deserialise from JSON
    article_text = example.article_text

    # Parse extraction output
    extracted = parse_extraction_output(pred.assay_info)

    # Compare extractions
    comparison = compare_extractions(extracted, ground_truth, article_text)

    # Calculate F1 metrics
    conservative_f1, primary_f1, optimistic_f1 = calculate_f1_metrics(comparison)

    # Use primary F1 as the main score
    score = primary_f1

    if pred_name is None:
        return score

    # Generate detailed feedback for GEPA
    feedback = generate_detailed_feedback(comparison, ground_truth)

    return dspy.Prediction(score=score, feedback=feedback)


def assay_metric_simple(example: dspy.Example, pred: dspy.Prediction, trace: Any = None) -> float:
    """Simple metric for evaluation (without feedback)."""
    return assay_metric_with_feedback(example, pred, trace, None, None)

---
## Section 12: Dataset Preparation for DSPy
---

In [15]:
def validate_hashable(value: Any, field_name: str) -> None:
    """Validate that a value is hashable. Raises ValueError if not."""
    try:
        hash(value)
    except TypeError as e:
        raise ValueError(f"Field '{field_name}' is not hashable: {type(value).__name__}. Error: {e}")


def create_dspy_examples(
    pmcid_list: List[str],
    xml_mapping: Dict[str, str],
    ground_truth_dir: Path,
    max_tokens: int = DEFAULT_MAX_TOKENS
) -> List[dspy.Example]:
    """
    Create DSPy Example objects for training/validation.

    Each Example contains:
        - pmcid: Article identifier (string)
        - article_text: Processed article text with token limit (string)
        - ground_truth: Flattened ground truth as JSON string (string)

    All fields are validated to be hashable before Example creation.
    """
    examples = []
    skipped = []
    total_tokens = 0
    hashability_errors = []

    for pmcid in pmcid_list:
        # Check XML file exists
        if pmcid not in xml_mapping:
            skipped.append((pmcid, "no XML file"))
            continue

        # Load ground truth
        gt_data = load_ground_truth(pmcid, ground_truth_dir)
        if gt_data is None:
            skipped.append((pmcid, "no ground truth"))
            continue

        # Flatten ground truth
        flattened_gt = flatten_ground_truth_assays(gt_data)

        # Prepare article text with token limit
        try:
            article_text, raw_xml, token_estimate = prepare_article_for_extraction(
                pmcid,
                xml_mapping[pmcid],
                max_tokens=max_tokens
            )
            total_tokens += token_estimate
        except Exception as e:
            skipped.append((pmcid, f"XML processing error: {e}"))
            continue

        # Ensure all fields are strings (hashable) #changed v1.4
        pmcid_str = str(pmcid)  #changed - explicit string conversion
        article_text_str = str(article_text)  #changed - explicit string conversion
        ground_truth_str = json.dumps(flattened_gt, sort_keys=True)  #changed - sort_keys for consistency

        # Validate hashability before creating Example #changed v1.4
        try:
            validate_hashable(pmcid_str, "pmcid")
            validate_hashable(article_text_str, "article_text")
            validate_hashable(ground_truth_str, "ground_truth")
        except ValueError as e:
            hashability_errors.append((pmcid, str(e)))
            print(f"WARNING: Hashability error for {pmcid}: {e}")
            continue

        # Create DSPy Example with validated hashable fields #changed v1.4
        try:
            example = dspy.Example(
                pmcid=pmcid_str,
                article_text=article_text_str,
                ground_truth=ground_truth_str
            ).with_inputs("article_text")

            examples.append(example)
        except Exception as e:
            hashability_errors.append((pmcid, f"Example creation error: {e}"))
            print(f"WARNING: Example creation failed for {pmcid}: {e}")
            continue

    print(f"Created {len(examples)} DSPy examples")
    print(f"Total estimated tokens: {total_tokens:,}")
    print(f"Average tokens per example: {total_tokens // len(examples) if examples else 0:,}")

    if skipped:
        print(f"Skipped {len(skipped)} documents: {skipped[:5]}...")

    if hashability_errors:
        print(f"WARNING: {len(hashability_errors)} hashability errors encountered")
        for pmcid, error in hashability_errors[:3]:
            print(f"  - {pmcid}: {error}")

    return examples

---
## Section 13: GEPA Logging Functions
---

In [16]:
def create_gepa_log_filepath(output_directory: str, experiment_id: str) -> str:
    """Generate filepath for GEPA log file."""
    return f"{output_directory}/{experiment_id}_gepa_log.txt"


def initialise_gepa_log(log_filepath: str, config: GEPAExperimentConfig) -> None:
    """Initialise the GEPA log file with experiment header."""
    header = f"""{'=' * 80}
GEPA ASSAY EXTRACTION OPTIMISATION LOG
{'=' * 80}

Experiment ID: {config.experiment_id}
Split: {config.split_name}
Extraction Model: {config.extraction_model_id}
Reflection Model: {config.reflection_model_id}
Num Threads: {config.num_threads}
Auto Budget: {config.auto_budget}
Started: {config.timestamp}

{'=' * 80}

"""
    with open(log_filepath, 'w', encoding='utf-8') as f:
        f.write(header)


def log_gepa_baseline(log_filepath: str, baseline_score: float, num_examples: int) -> None:
    """Log baseline (pre-optimisation) results."""
    entry = f"""
{'=' * 80}
BASELINE EVALUATION (Pre-Optimisation)
{'=' * 80}
Number of examples: {num_examples}
Baseline F1 Score: {baseline_score:.4f}
Timestamp: {datetime.now().isoformat()}

"""
    with open(log_filepath, 'a', encoding='utf-8') as f:
        f.write(entry)


def log_gepa_completion(
    log_filepath: str,
    final_score: float,
    improvement: float,
    total_iterations: int,
    optimised_program: dspy.Module
) -> None:
    """Log GEPA optimisation completion."""
    entry = f"""
{'=' * 80}
GEPA OPTIMISATION COMPLETE
{'=' * 80}
Final F1 Score: {final_score:.4f}
Improvement: {improvement:+.4f}
Total Iterations: {total_iterations}
Completed: {datetime.now().isoformat()}

OPTIMISED SIGNATURE:
"""
    with open(log_filepath, 'a', encoding='utf-8') as f:
        f.write(entry)

        for name, predictor in optimised_program.named_predictors():
            f.write(f"\n--- Predictor: {name} ---\n")
            if hasattr(predictor, 'signature'):
                f.write(f"{predictor.signature}\n")

        f.write(f"\n{'=' * 80}\n")


def log_signature_evolution(log_filepath: str, stage: str, program: dspy.Module) -> None:
    """Log the current state of program signatures."""
    entry = f"""
--- SIGNATURE STATE: {stage} ---
Timestamp: {datetime.now().isoformat()}
"""
    with open(log_filepath, 'a', encoding='utf-8') as f:
        f.write(entry)

        for name, predictor in program.named_predictors():
            f.write(f"\nPredictor: {name}\n")
            if hasattr(predictor, 'signature'):
                sig = predictor.signature
                f.write(f"Docstring: {sig.__doc__}\n")
                f.write(f"Input fields: {list(sig.input_fields.keys())}\n")
                f.write(f"Output fields: {list(sig.output_fields.keys())}\n")

        f.write("\n")

---
## Section 14: Cost Tracking
---

In [17]:
def calculate_cost(model_id: str, input_tokens: int, output_tokens: int) -> float:
    """Calculate cost in USD for API usage."""
    pricing = MODEL_PRICING.get(model_id, {"input": 0, "output": 0})
    input_cost = (input_tokens / 1_000_000) * pricing["input"]
    output_cost = (output_tokens / 1_000_000) * pricing["output"]
    return input_cost + output_cost


def estimate_gepa_cost(
    num_train_examples: int,
    num_val_examples: int,
    auto_budget: str,
    avg_input_tokens: int = 8000,
    avg_output_tokens: int = 2000
) -> Dict[str, float]:
    """Estimate GEPA optimisation cost."""
    budget_multipliers = {
        "light": 12,
        "medium": 25,
        "heavy": 50
    }

    multiplier = budget_multipliers.get(auto_budget, 12)
    total_examples = num_train_examples + num_val_examples
    estimated_calls = total_examples * multiplier

    # Extraction model cost (Haiku)
    extraction_cost = calculate_cost(
        "claude-haiku-4.5",
        estimated_calls * avg_input_tokens,
        estimated_calls * avg_output_tokens
    )

    # Reflection model cost (Sonnet)
    reflection_calls = estimated_calls // 10
    reflection_cost = calculate_cost(
        "claude-sonnet-4",
        reflection_calls * 4000,
        reflection_calls * 4000
    )

    return {
        "extraction_cost_usd": extraction_cost,
        "reflection_cost_usd": reflection_cost,
        "total_cost_usd": extraction_cost + reflection_cost,
        "estimated_extraction_calls": estimated_calls,
        "estimated_reflection_calls": reflection_calls
    }


def print_cost_estimate(cost_estimate: Dict[str, float]) -> None:
    """Print formatted cost estimate."""
    print("\n" + "=" * 50)
    print("ESTIMATED GEPA OPTIMISATION COST")
    print("=" * 50)
    print(f"Extraction calls (Haiku): ~{cost_estimate['estimated_extraction_calls']:,}")
    print(f"Extraction cost: ${cost_estimate['extraction_cost_usd']:.2f}")
    print(f"Reflection calls (Sonnet): ~{cost_estimate['estimated_reflection_calls']:,}")
    print(f"Reflection cost: ${cost_estimate['reflection_cost_usd']:.2f}")
    print(f"TOTAL ESTIMATED COST: ${cost_estimate['total_cost_usd']:.2f}")
    print("=" * 50 + "\n")

---
## Section 15: GEPA Optimisation Runner
---

In [18]:
def run_gepa_optimisation(
    train_examples: List[dspy.Example],
    val_examples: List[dspy.Example],
    config: GEPAExperimentConfig,
    extraction_lm: dspy.LM,
    reflection_lm: dspy.LM
) -> Tuple[dspy.Module, Dict[str, Any]]:
    """
    Run GEPA optimisation for assay extraction.

    Returns:
        Tuple of (optimised_program, results_dict)
    """
    # Initialise logging
    log_filepath = create_gepa_log_filepath(config.output_directory, config.experiment_id)
    initialise_gepa_log(log_filepath, config)

    # Configure DSPy with extraction model
    # Note: cache=False prevents hashability errors with complex Example/Prediction objects #changed
    dspy.configure(lm=extraction_lm, cache=False)  #changed - disable caching

    # Create base program
    base_program = AssayExtractor()

    # Log initial signature
    log_signature_evolution(log_filepath, "INITIAL", base_program)

    # Run baseline evaluation
    print("Running baseline evaluation...")
    # Evaluate with provide_traceback=True for detailed error information #changed v1.4
    baseline_evaluate = dspy.Evaluate(
        devset=val_examples,
        metric=assay_metric_simple,
        num_threads=config.num_threads,
        display_progress=True,
        provide_traceback=True  #changed - enables full traceback on errors
    )
    baseline_result = baseline_evaluate(base_program)
    baseline_score = baseline_result.score / 100.0

    log_gepa_baseline(log_filepath, baseline_score, len(val_examples))
    print(f"Baseline F1 Score: {baseline_score:.4f}")

    # Estimate cost
    cost_estimate = estimate_gepa_cost(
        len(train_examples),
        len(val_examples),
        config.auto_budget
    )
    print_cost_estimate(cost_estimate)

    # Confirm before proceeding
    confirm = input("Proceed with GEPA optimisation? (yes/no): ")
    if confirm.lower() != "yes":
        print("Optimisation cancelled.")
        return base_program, {"status": "cancelled", "baseline_score": baseline_score}

    # Create GEPA optimiser
    print("\nInitialising GEPA optimiser...")
    # Create experiment-specific GEPA log directory #changed
    experiment_gepa_log_dir = str(GEPA_LOG_DIR / config.experiment_id)  #changed
    Path(experiment_gepa_log_dir).mkdir(parents=True, exist_ok=True)  #changed
    print(f"GEPA internal logs will be saved to: {experiment_gepa_log_dir}")  #changed

    optimizer = dspy.GEPA(
        metric=assay_metric_with_feedback,
        auto=config.auto_budget,
        num_threads=config.num_threads,
        track_stats=True,
        use_merge=False,
        reflection_lm=reflection_lm,
        log_dir=experiment_gepa_log_dir  #changed - enables GEPA's elaborate logging and checkpoint resuming
    )

    # Run optimisation with detailed error handling #changed v1.4
    print("\nStarting GEPA optimisation...")
    start_time = time.time()

    try:
        optimised_program = optimizer.compile(
            base_program,
            trainset=train_examples,
            valset=val_examples
        )
    except TypeError as e:
        # Catch hashability errors and provide detailed traceback #changed v1.4
        import traceback
        error_msg = f"\n{'='*80}\nGEPA OPTIMISATION ERROR\n{'='*80}\n"
        error_msg += f"Error Type: {type(e).__name__}\n"
        error_msg += f"Error Message: {str(e)}\n"
        error_msg += f"\nFull Traceback:\n{traceback.format_exc()}\n"
        error_msg += "\nThis error typically occurs when DSPy attempts to hash Example or Prediction objects.\n"
        error_msg += "Check that all Example fields are primitive hashable types (str, int, float, bool, tuple).\n"
        error_msg += f"{'='*80}\n"
        print(error_msg)

        # Also write to log file
        with open(log_filepath, 'a', encoding='utf-8') as f:
            f.write(error_msg)

        return base_program, {
            "status": "error",
            "error_type": type(e).__name__,
            "error_message": str(e),
            "baseline_score": baseline_score
        }

    optimisation_time = time.time() - start_time

    # Log optimised signature
    log_signature_evolution(log_filepath, "OPTIMISED", optimised_program)

    # Evaluate optimised program
    print("\nEvaluating optimised program...")
    final_result = baseline_evaluate(optimised_program)
    final_score = final_result.score / 100.0

    improvement = final_score - baseline_score

    # Log completion
    log_gepa_completion(
        log_filepath,
        final_score,
        improvement,
        getattr(optimizer, 'total_iterations', 0),
        optimised_program
    )

    # Prepare results
    results = {
        "status": "completed",
        "baseline_score": baseline_score,
        "final_score": final_score,
        "improvement": improvement,
        "optimisation_time_seconds": optimisation_time,
        "num_train_examples": len(train_examples),
        "num_val_examples": len(val_examples),
        "cost_estimate": cost_estimate,
        "config": {
            "experiment_id": config.experiment_id,
            "split_name": config.split_name,
            "auto_budget": config.auto_budget,
            "num_threads": config.num_threads
        }
    }

    # Print summary
    print("\n" + "=" * 50)
    print("GEPA OPTIMISATION COMPLETE")
    print("=" * 50)
    print(f"Baseline F1: {baseline_score:.4f}")
    print(f"Final F1: {final_score:.4f}")
    print(f"Improvement: {improvement:+.4f}")
    print(f"Time: {optimisation_time:.1f} seconds")
    print("=" * 50)

    return optimised_program, results


---
## Section 16: Save and Load Optimised Program
---

In [19]:
def save_optimised_program(
    program: dspy.Module,
    results: Dict[str, Any],
    output_directory: str,
    experiment_id: str
) -> Tuple[str, str]:
    """Save optimised program and results."""
    # Save program using DSPy's save method
    program_filepath = f"{output_directory}/{experiment_id}_optimised_program.json"
    program.save(program_filepath)
    print(f"Optimised program saved to: {program_filepath}")

    # Save results
    results_filepath = f"{output_directory}/{experiment_id}_results.json"
    with open(results_filepath, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2, default=str)
    print(f"Results saved to: {results_filepath}")

    return program_filepath, results_filepath


def load_optimised_program(program_filepath: str) -> dspy.Module:
    """Load a previously optimised program."""
    program = AssayExtractor()
    program.load(program_filepath)
    print(f"Loaded optimised program from: {program_filepath}")
    return program

---
## Section 17: Load Data and Prepare Experiments
---

In [20]:
# Load stratified splits
splits_data = load_stratified_splits(str(STRATIFIED_SPLITS_FILE))

Loaded stratified splits:
  Total documents: 227
  Holdout size: 45
  Validation size: 35
  Training pool size: 147
  Split 10% size: 18
  Split 20% size: 37
  Split 30% size: 54


In [21]:
# Get PMCIDs for each split
split_10_pmcids = get_split_pmcids(splits_data, "split_10")
split_20_pmcids = get_split_pmcids(splits_data, "split_20")
split_30_pmcids = get_split_pmcids(splits_data, "split_30")
validation_pmcids = get_split_pmcids(splits_data, "validation_set")

print(f"Split 10% size: {len(split_10_pmcids)}")
print(f"Split 20% size: {len(split_20_pmcids)}")
print(f"Split 30% size: {len(split_30_pmcids)}")
print(f"Validation set size: {len(validation_pmcids)}")

Split 10% size: 18
Split 20% size: 37
Split 30% size: 54
Validation set size: 35


In [22]:
# Build XML mapping for all required PMCIDs (handles PMCID_Date.xml format)
all_pmcids = list(set(split_10_pmcids + split_20_pmcids + split_30_pmcids + validation_pmcids))
xml_mapping = load_pmcid_to_xml_mapping(str(XML_DIR), all_pmcids)

Built XML mapping for 89/89 PMCIDs


In [23]:
# Create DSPy examples for 10% split and validation
print("\nCreating training examples (10% split)...")
train_examples_10 = create_dspy_examples(split_10_pmcids, xml_mapping, GROUND_TRUTH_DIR)

print("\nCreating validation examples...")
val_examples = create_dspy_examples(validation_pmcids, xml_mapping, GROUND_TRUTH_DIR)


Creating training examples (10% split)...
Created 18 DSPy examples
Total estimated tokens: 158,865
Average tokens per example: 8,825

Creating validation examples...
Created 35 DSPy examples
Total estimated tokens: 303,249
Average tokens per example: 8,664


---
## Section 18: Configure Language Models
---

In [24]:
# Configure extraction model (Haiku 4.5)
extraction_lm = dspy.LM(
    DSPY_MODEL_STRINGS["claude-haiku-4.5"],
    temperature=0.0,
    max_tokens=8192
)

# Configure reflection model (Sonnet 4)
reflection_lm = dspy.LM(
    DSPY_MODEL_STRINGS["claude-sonnet-4"],
    temperature=1.0,
    max_tokens=16000
)

print(f"Extraction model: {DSPY_MODEL_STRINGS['claude-haiku-4.5']}")
print(f"Reflection model: {DSPY_MODEL_STRINGS['claude-sonnet-4']}")

Extraction model: anthropic/claude-haiku-4-5-20251001
Reflection model: anthropic/claude-sonnet-4-20250514


---
## Section 19: Run GEPA Optimisation - 10% Split
---

In [25]:
# Create experiment configuration for 10% split
config_10 = GEPAExperimentConfig(
    experiment_id="gepa_assay_stage0_split10_v1",
    split_name="split_10",
    extraction_model_id="claude-haiku-4.5",
    reflection_model_id="claude-sonnet-4",
    num_threads=GEPA_CONFIG["num_threads"],
    auto_budget=GEPA_CONFIG["auto"],
    output_directory=str(OUTPUT_DIR)
)

print(f"Experiment ID: {config_10.experiment_id}")
print(f"Training examples: {len(train_examples_10)}")
print(f"Validation examples: {len(val_examples)}")

Experiment ID: gepa_assay_stage0_split10_v1
Training examples: 18
Validation examples: 35


In [26]:
# Run GEPA optimisation for 10% split
optimised_program_10, results_10 = run_gepa_optimisation(
    train_examples=train_examples_10,
    val_examples=val_examples,
    config=config_10,
    extraction_lm=extraction_lm,
    reflection_lm=reflection_lm
)

Running baseline evaluation...
Average Metric: 0.00 / 25 (0.0%):  71%|███████▏  | 25/35 [02:28<01:01,  6.16s/it]Warning: Could not parse JSON from output: {
  "Seq1": {
    "invA": "positive",
    "Serotype": "Salmonella enterica subsp. enterica",
    "Sequence_Similarity": "98%",
    "AST_Ampicillin": "resistant",
    "AST_Oxy-tetracycline": "resistant...
Average Metric: 0.00 / 35 (0.0%): 100%|██████████| 35/35 [03:38<00:00,  6.23s/it]
Baseline F1 Score: 0.0000

ESTIMATED GEPA OPTIMISATION COST
Extraction calls (Haiku): ~636
Extraction cost: $11.45
Reflection calls (Sonnet): ~63
Reflection cost: $4.54
TOTAL ESTIMATED COST: $15.98

Proceed with GEPA optimisation? (yes/no): yes

Initialising GEPA optimiser...
GEPA internal logs will be saved to: /content/drive/MyDrive/AI6129/assay/gepa_experiments/gepa_internal_logs/gepa_assay_stage0_split10_v1

Starting GEPA optimisation...


GEPA Optimization:   0%|          | 0/520 [00:00<?, ?rollouts/s]

  "Seq1": {
    "invA": "positive",
    "Serotype": "Salmonella enterica subsp. enterica",
    "Sequence_Similarity": "98%",
    "AST_Ampicillin": "resistant",
    "AST_Oxy-tetracycline": "resistant...


GEPA Optimization:   7%|▋         | 35/520 [00:01<00:22, 21.74rollouts/s]

Average Metric: 0.00 / 3 (0.0%): 100%|██████████| 3/3 [00:11<00:00,  3.93s/it]


GEPA Optimization:   8%|▊         | 41/520 [00:52<13:40,  1.71s/rollouts]

Average Metric: 0.00 / 3 (0.0%): 100%|██████████| 3/3 [00:27<00:00,  9.27s/it]
  "CFSAN033848": {"MLST": "ST19", "serotype": "Typhimurium", "AMR": ["acrA", "acrB", "macA", "macB", "mdtK", "emrA", "emrB", "emrR", "tolC", "mdsA", "mdsB"], "Prophages": ["Gifsy-2", "Salmon_118970_...


KeyboardInterrupt: 

In [ ]:
# Save optimised program and results
if results_10.get("status") == "completed":
    program_path_10, results_path_10 = save_optimised_program(
        optimised_program_10,
        results_10,
        str(OUTPUT_DIR),
        config_10.experiment_id
    )
    print(f"\nOptimised program saved: {program_path_10}")
    print(f"Results saved: {results_path_10}")

---
## Section 20: View Optimised Signature
---

In [ ]:
# Display the optimised signature
if results_10.get("status") == "completed":
    print("\n" + "=" * 80)
    print("OPTIMISED SIGNATURE")
    print("=" * 80)

    for name, predictor in optimised_program_10.named_predictors():
        print(f"\nPredictor: {name}")
        if hasattr(predictor, 'signature'):
            sig = predictor.signature
            print(f"\nDocstring:\n{sig.__doc__}")
            print(f"\nInput fields: {list(sig.input_fields.keys())}")
            print(f"Output fields: {list(sig.output_fields.keys())}")

---
## Section 21: Results Summary
---

In [ ]:
# Print final results summary
print("\n" + "=" * 80)
print("GEPA STAGE 0 PILOT - 10% SPLIT RESULTS")
print("=" * 80)

if results_10.get("status") == "completed":
    print(f"\nExperiment ID: {results_10['config']['experiment_id']}")
    print(f"Split: {results_10['config']['split_name']}")
    print(f"\nTraining examples: {results_10['num_train_examples']}")
    print(f"Validation examples: {results_10['num_val_examples']}")
    print(f"\nBaseline F1: {results_10['baseline_score']:.4f}")
    print(f"Final F1: {results_10['final_score']:.4f}")
    print(f"Improvement: {results_10['improvement']:+.4f}")
    print(f"\nOptimisation time: {results_10['optimisation_time_seconds']:.1f} seconds")
    print(f"Estimated cost: ${results_10['cost_estimate']['total_cost_usd']:.2f}")
else:
    print(f"\nStatus: {results_10.get('status', 'unknown')}")
    if 'baseline_score' in results_10:
        print(f"Baseline F1: {results_10['baseline_score']:.4f}")

print("\n" + "=" * 80)

---
## Section 22: Next Steps - 20% and 30% Splits

To run the 20% and 30% splits, execute the following cells after the 10% split completes.

**Important:** Run each split sequentially to avoid rate limiting issues.
---

In [ ]:
# Uncomment to run 20% split
# print("\nCreating training examples (20% split)...")
# train_examples_20 = create_dspy_examples(split_20_pmcids, xml_mapping, GROUND_TRUTH_DIR)
#
# config_20 = GEPAExperimentConfig(
#     experiment_id="gepa_assay_stage0_split20_v1",
#     split_name="split_20",
#     extraction_model_id="claude-haiku-4.5",
#     reflection_model_id="claude-sonnet-4",
#     num_threads=GEPA_CONFIG["num_threads"],
#     auto_budget=GEPA_CONFIG["auto"],
#     output_directory=str(OUTPUT_DIR)
# )
#
# optimised_program_20, results_20 = run_gepa_optimisation(
#     train_examples=train_examples_20,
#     val_examples=val_examples,
#     config=config_20,
#     extraction_lm=extraction_lm,
#     reflection_lm=reflection_lm
# )
#
# if results_20.get("status") == "completed":
#     save_optimised_program(optimised_program_20, results_20, str(OUTPUT_DIR), config_20.experiment_id)

In [ ]:
# Uncomment to run 30% split
# print("\nCreating training examples (30% split)...")
# train_examples_30 = create_dspy_examples(split_30_pmcids, xml_mapping, GROUND_TRUTH_DIR)
#
# config_30 = GEPAExperimentConfig(
#     experiment_id="gepa_assay_stage0_split30_v1",
#     split_name="split_30",
#     extraction_model_id="claude-haiku-4.5",
#     reflection_model_id="claude-sonnet-4",
#     num_threads=GEPA_CONFIG["num_threads"],
#     auto_budget=GEPA_CONFIG["auto"],
#     output_directory=str(OUTPUT_DIR)
# )
#
# optimised_program_30, results_30 = run_gepa_optimisation(
#     train_examples=train_examples_30,
#     val_examples=val_examples,
#     config=config_30,
#     extraction_lm=extraction_lm,
#     reflection_lm=reflection_lm
# )
#
# if results_30.get("status") == "completed":
#     save_optimised_program(optimised_program_30, results_30, str(OUTPUT_DIR), config_30.experiment_id)

---
## Notes and Design Decisions

### Key Design Decisions (Stage 0 Pilot)

1. **Single-Module Approach**: Using a single `AssayExtractor` module for simplicity in the pilot phase.

2. **UP Classification**: Implemented Uncertain Positive (UP) category to distinguish between true hallucinations and potential ground truth gaps.

3. **Three F1 Metrics**:
   - Conservative F1: Treats UP as FP (lower bound)
   - Primary F1: Excludes UP (recommended metric)
   - Optimistic F1: Treats UP as TP (upper bound)

4. **Rate Limiting**: Using `num_threads=2` and `auto="light"` to prevent API rate limit issues.

5. **XML Mapping**: Handles `{PMCID}_{Date}.xml` filename format using glob and filename splitting.

6. **Token Truncation**: Matches accession extraction pipeline with 100k token limit and 70/30 front/back truncation.

### Files Generated

- `{experiment_id}_optimised_program.json`: Saved DSPy program
- `{experiment_id}_results.json`: Experiment results and metrics
- `{experiment_id}_gepa_log.txt`: Detailed GEPA iteration log

### Next Steps After Stage 0

1. Analyse results across 10%/20%/30% splits to determine optimal training data size
2. Run inference on test set using separate notebook with optimised prompts
3. Validate on holdout set for final evaluation
4. Extend to Hepatitis A/E pathogens for cross-pathogen validation
---